## Feature Loading

In [ ]:
!pip install opencv-python
!pip install scikit-image
!pip install pyefd
!pip install mahotas

In [14]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

from preproc import (
    load_and_normalize,
    segment_fruit,
    refine_mask,
    compute_shape_features,
    compute_colour_features,
    build_feature_vector
)

VALID_EXTS = {".jpg", ".jpeg", ".png"}

def process_single_image(image_path):
    img = load_and_normalize(str(image_path))
    raw_mask = segment_fruit(img)
    mask_refined = refine_mask(raw_mask)
    shape_feats = compute_shape_features(mask_refined)
    colour_feats = compute_colour_features(img, mask_refined)
    feature_vector = build_feature_vector(shape_feats, colour_feats)
    return feature_vector


def build_feature_dataset(root_dir):
    """
    Loop through each subfolder in root_dir, process all .jpg/.jpeg/.png images,
    and save the resulting feature vectors to a CSV.
    """
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Root directory does not exist: {root}")

    rows = []
    errors = []

    # only immediate subfolders
    subfolders = sorted([p for p in root.iterdir() if p.is_dir()])

    print(f"Found {len(subfolders)} subfolders.")

    for subfolder in subfolders:
        image_files = sorted(
            [p for p in subfolder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS]
        )

        print(f"\nProcessing folder: {subfolder.name} ({len(image_files)} images)")

        for image_path in image_files:
            try:
                fv = process_single_image(image_path)

                row = {
                    "subfolder": subfolder.name,
                    "filename": image_path.name,
                    "filepath": str(image_path)
                }

                # add each feature as its own column
                for i, val in enumerate(fv):
                    row[f"feature_{i:03d}"] = float(val)

                rows.append(row)

            except Exception as e:
                errors.append((subfolder.name, image_path.name, str(e)))
                print(f"  Failed: {image_path.name} -> {e}")

    df = pd.DataFrame(rows)

    print("\nDone.")
    print(f"Successful images: {len(df)}")
    print(f"Failed images: {len(errors)}")

    return df, errors


In [ ]:
# Feature vectors with errors printed

df_features, failed = build_feature_dataset(
        root_dir="/content/drive/MyDrive/data/fruit_pics_all"
    )

print("\nFeature table shape:", df_features.shape)

if len(df_features) > 0:
  feature_cols = [c for c in df_features.columns if c.startswith("feature_")]
  print("Number of extracted features per image:", len(feature_cols))
  print("First image, first 10 features:")
  print(df_features.loc[0, feature_cols[:10]].values)

if failed:
  print("\nSome images failed:")
  for item in failed[:10]:
    print(item)

#df_features.to_csv("fruit_features.csv", index=False)

## Data Creation

In [16]:
import pandas as pd
from pathlib import Path

# load data
fruit_features = pd.read_csv("fruit_features.csv")
fruit_labels_metadata = pd.read_csv("fruit_labels_metadata.csv")

# create a clean image_id in fruit_features by removing the extension
fruit_features["image_id"] = fruit_features["filename"].apply(lambda x: Path(x).stem)

# keep only the columns you want from labels file
labels_only = fruit_labels_metadata[["image_id", "is_fruit"]]

# merge
merged = fruit_features.merge(labels_only, on="image_id", how="left")

# inspect
print(merged[["filename", "is_fruit","feature_000", "feature_001", "feature_002"]].head())
print(merged.shape)

           filename  is_fruit  feature_000  feature_001  feature_002
0  01300169_001.jpg      True       1211.5   132.610168     3.809913
1  01300169_002.jpg      True        738.0   105.254837     3.874484
2  01300169_003.jpg      True       2413.5   326.835571     6.652818
3  01300169_004.jpg      True       1021.0   121.053825     3.788486
4  01300169_005.jpg      True       1823.0   166.994949     3.911200
(374, 99)


In [17]:
df_features = merged.copy()
df_features.to_csv("fruit_train.csv", index=False)

## Splits

In [4]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Grouped splits by subfolder instead of indiviual pictures
feature_cols = [c for c in df_features.columns if c.startswith("feature_")]
X = df_features[feature_cols]
y = df_features["is_fruit"]
groups = df_features["subfolder"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train groups:", groups_train.nunique())
print("Test groups:", groups_test.nunique())

Train shape: (297, 94)
Test shape: (77, 94)
Train groups: 15
Test groups: 4


## Helper Functions

In [5]:
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

def gridSearch_fit_predict(model, param_grid, X_tr, y_tr, X_val, y_val, groups):
    grid = GridSearchCV(
        model,
        param_grid,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    grid.fit(X_tr, y_tr, groups=groups)
    best_model = grid.best_estimator_
    print(grid.best_params_)

    y_pred = best_model.predict(X_val)
    print(classification_report(y_val, y_pred))
    return best_model

def cv_performance(model, X, y, groups):
  y_pred_cv = cross_val_predict(
    model,
    X,
    y,
    groups=groups,
    cv=cv,
    method="predict"
)

  print("Accuracy:", (y_pred_cv == y).mean())
  print("Balanced accuracy:", balanced_accuracy_score(y, y_pred_cv))
  print("F1:", f1_score(y, y_pred_cv))
  print(confusion_matrix(y, y_pred_cv))
  print(classification_report(y, y_pred_cv))
  return y_pred_cv


## Random Forest

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

best_rf = gridSearch_fit_predict(rf, param_grid, X_train, y_train, X_test, y_test, groups_train)

{'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 100}
              precision    recall  f1-score   support

       False       0.72      0.92      0.81        37
        True       0.90      0.68      0.77        40

    accuracy                           0.79        77
   macro avg       0.81      0.80      0.79        77
weighted avg       0.82      0.79      0.79        77



In [17]:
y_pred_cv_rf = cv_performance(best_rf, X, y, groups)

Accuracy: 0.8235294117647058
Balanced accuracy: 0.8239130434782609
F1: 0.8216216216216217
[[156  28]
 [ 38 152]]
              precision    recall  f1-score   support

       False       0.80      0.85      0.83       184
        True       0.84      0.80      0.82       190

    accuracy                           0.82       374
   macro avg       0.82      0.82      0.82       374
weighted avg       0.82      0.82      0.82       374



In [19]:
import pandas as pd

feature_names = [c for c in df_features.columns if c.startswith("feature_")]

importances = best_rf.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(importance_df.head(20))

        feature  importance
5   feature_005    0.062909
15  feature_015    0.051322
2   feature_002    0.047503
3   feature_003    0.044630
49  feature_049    0.043741
89  feature_089    0.041377
85  feature_085    0.032135
4   feature_004    0.031454
48  feature_048    0.027248
83  feature_083    0.026979
69  feature_069    0.023232
71  feature_071    0.023051
70  feature_070    0.022827
65  feature_065    0.022616
75  feature_075    0.021308
84  feature_084    0.019970
88  feature_088    0.018283
10  feature_010    0.018081
66  feature_066    0.018073
81  feature_081    0.015162


## XGBOOST

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.7, 1.0]
}

best_gb = gridSearch_fit_predict(gb, param_grid_gb, X_train, y_train, X_test, y_test, groups_train)


{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.7}
              precision    recall  f1-score   support

       False       0.72      0.84      0.78        37
        True       0.82      0.70      0.76        40

    accuracy                           0.77        77
   macro avg       0.77      0.77      0.77        77
weighted avg       0.77      0.77      0.77        77



In [27]:
y_pred_cv_xgb = cv_performance(best_gb, X, y, groups)

Accuracy: 0.8609625668449198
Balanced accuracy: 0.8611842105263158
F1: 0.8609625668449198
[[161  23]
 [ 29 161]]
              precision    recall  f1-score   support

       False       0.85      0.88      0.86       184
        True       0.88      0.85      0.86       190

    accuracy                           0.86       374
   macro avg       0.86      0.86      0.86       374
weighted avg       0.86      0.86      0.86       374



## CatBoost

In [12]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 26.3 MB/s eta 0:00:00


In [14]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    verbose=0,
    random_state=42
)

param_grid_cat = {
    "iterations": [100, 200, 300],
    "depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7]
}

best_cat = gridSearch_fit_predict(cat, param_grid_cat, X_train, y_train, X_test, y_test, groups_train)

{'depth': 8, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.1}
              precision    recall  f1-score   support

       False       0.76      0.86      0.81        37
        True       0.86      0.75      0.80        40

    accuracy                           0.81        77
   macro avg       0.81      0.81      0.81        77
weighted avg       0.81      0.81      0.80        77



In [14]:
print(best_cat.get_params())

{'verbose': 0, 'random_state': 42, 'depth': 8, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.1}


In [15]:
y_pred_cv_cat = cv_performance(best_cat, X, y, groups)

Accuracy: 0.8663101604278075
Balanced accuracy: 0.8667906178489703
F1: 0.8641304347826086
[[165  19]
 [ 31 159]]
              precision    recall  f1-score   support

       False       0.84      0.90      0.87       184
        True       0.89      0.84      0.86       190

    accuracy                           0.87       374
   macro avg       0.87      0.87      0.87       374
weighted avg       0.87      0.87      0.87       374



In [18]:
best_cat.save_model("catboost_model.cbm")

## SVM

In [43]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# SVM pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", random_state=42))
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.001, 0.01, 0.1, 1]
}

best_svm = gridSearch_fit_predict(pipe, param_grid, X_train, y_train, X_test, y_test, groups_train)

{'svm__C': 0.1, 'svm__gamma': 'scale'}
              precision    recall  f1-score   support

       False       0.67      0.84      0.75        37
        True       0.81      0.62      0.70        40

    accuracy                           0.73        77
   macro avg       0.74      0.73      0.73        77
weighted avg       0.74      0.73      0.72        77



In [44]:
y_pred_cv_svm = cv_performance(best_svm, X, y, groups)

Accuracy: 0.7807486631016043
Balanced accuracy: 0.7806064073226544
F1: 0.7853403141361257
[[142  42]
 [ 40 150]]
              precision    recall  f1-score   support

       False       0.78      0.77      0.78       184
        True       0.78      0.79      0.79       190

    accuracy                           0.78       374
   macro avg       0.78      0.78      0.78       374
weighted avg       0.78      0.78      0.78       374



## Error Analysis

In [45]:
import pandas as pd

# Mean of correctly classifed pictures by subfolder (using CatBoost classifier)
df_eval = df_features.copy()
df_eval["y_true"] = y
df_eval["y_pred"] = y_pred_cv_cat
df_eval["correct"] = (df_eval["y_true"] == df_eval["y_pred"]).astype(int)

folder_perf = (
    df_eval.groupby("subfolder")["correct"]
    .agg(["mean", "count"])
    .sort_values("mean")
)


print(folder_perf)

               mean  count
subfolder                 
1430300    0.700000     20
1471890    0.736842     19
14245930   0.736842     19
1475980    0.750000     20
1300169    0.789474     19
1439343    0.800000     20
1674115    0.800000     20
1658436    0.800000     20
1663557    0.842105     19
1610963    0.894737     19
1650064    0.900000     20
1495161    0.900000     20
1483309    0.950000     20
1625398    0.950000     20
1452427    0.950000     20
1460705    0.950000     20
1570623    1.000000     20
1616148    1.000000     19
1666596    1.000000     20


In [46]:
# Confusion matrix for highest performing classifier
pd.crosstab(df_eval["y_true"], df_eval["y_pred"], margins=True)

y_pred,False,True,All
y_true,,,
False,165,19,184
True,31,159,190
All,196,178,374


In [9]:
# Difference of feature means of correctly classified observations vs. incorrectly classified observations
errors = df_eval[df_eval["correct"] == 0]
correct = df_eval[df_eval["correct"] == 1]

feature_cols = [c for c in df_features.columns if c.startswith("feature_")]

diff = (errors[feature_cols].mean() - correct[feature_cols].mean()).sort_values(key=abs, ascending=False)

print("Features       Difference")
print(diff.head(15))

Features       Difference
feature_000    108.855309
feature_089    -19.094641
feature_090     17.442271
feature_091      6.113908
feature_093     -5.601532
feature_001      4.284884
feature_047      4.279261
feature_092     -2.023401
feature_051      1.814858
feature_050      1.747156
feature_049     -0.891315
feature_052      0.820630
feature_054      0.656113
feature_056      0.358429
feature_088     -0.351928
dtype: float64
